# PIPE2D-1391-01 verification

Manual verification for the QA rebuild Phase 0/1 branch, to run against a real
Butler before merging.

**This notebook is read-only.** It opens the Butler with `writeable=False`, runs no
`pipetask run`, and writes nothing to any collection or repository. The only files it
creates are optional plot images under `OUTPUT_DIR`, and only if you ask for them in
section 5. Where a check genuinely needs a pipeline run, the notebook prints the command
for you to run yourself rather than running it.

## What each section establishes

| Section | Question | Needs |
|---|---|---|
| 1 | Does everything import, and does the pipeline build? | stack only |
| 2 | Do the stack-dependent tests pass? | stack only |
| 3 | **Did the registry migration move any verdict?** | a collection with `iqQaMetrics` |
| 4 | Is the new per-species dataset present and readable? | a collection from this branch |
| 5 | Do the extracted plotting functions still draw? | a collection with `dmQaResidualData` |
| 6 | Do the golden `known_good` visits pass? | a collection covering Run25 |
| 7 | Does the threshold CLI run against a real Butler? | a collection with `iqQaMetrics` |

Section 3 is the one that matters most: the ticket promises **no change to any QA
verdict**, and section 3 tests exactly that, on real numbers, without running anything.

## Configuration

Set these, then run the notebook top to bottom.

In [ ]:
# --- Butler ---------------------------------------------------------------
BUTLER_REPO = "/path/to/butler"

# Any collection holding iqQaMetrics. It does NOT need to come from this branch:
# section 3a only reads the measured values, which this branch does not touch, so an
# old collection reduced months ago on main works perfectly. Section 3b additionally
# reads the stored qaStatus and says so when that makes it circular.
COLLECTION = "u/you/qa-run"

# Optional: a second collection to diff against, if you did run the pipeline twice.
# Leave as None to skip the direct two-collection comparison in section 3b.
BASELINE_COLLECTION = None

# Optional: restrict every query to these visits. None means "whatever is there".
VISITS = None  # e.g. [133025, 133028, 133031, 133034, 133037, 133040]

# --- Local ----------------------------------------------------------------
OUTPUT_DIR = "pipe2d1391-verification"  # only written to if you opt in, in section 5
REPO_ROOT = ".."                        # path to the drp_qa checkout from this notebook

In [ ]:
import subprocess
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path(REPO_ROOT).resolve()
sys.path.insert(0, str(REPO_ROOT / "python"))

RESULTS = {}  # section -> (ok, message); summarised at the end


def record(section, ok, message):
    """Record a result and print it."""
    RESULTS[section] = (ok, message)
    mark = "PASS" if ok else ("SKIP" if ok is None else "FAIL")
    print(f"[{mark}] {section}: {message}")


def skip(section, message):
    record(section, None, message)


print(f"drp_qa checkout: {REPO_ROOT}")

## 1. Imports and pipeline build

Exercises the plotting extraction and the `FitStats` move. A broken re-export shim fails
here rather than part-way through a reduction.

In [ ]:
try:
    import pfs.drp.qa.dmCombinedResiduals  # noqa: F401
    import pfs.drp.qa.dmResiduals  # noqa: F401
    import pfs.drp.qa.imageQualityQa  # noqa: F401
    import pfs.drp.qa.plotting  # noqa: F401
    from pfs.drp.qa.iqQaPlots import plotIqTimeSeries  # legacy shim  # noqa: F401
    from pfs.drp.qa.utils.plotting import detector_palette  # legacy shim  # noqa: F401
except Exception as exc:  # noqa: BLE001
    record("1 imports", False, f"{type(exc).__name__}: {exc}")
else:
    record("1 imports", True, "all modules and legacy shims import")

In [ ]:
# --show tasks does not consume the butler, so passing -b is rejected -- and
# without -b the graph cannot resolve the PFS dimensions arm and spectrograph,
# which live in the repository's dimension config rather than the default
# universe. --show pipeline-graph is one of the forms that does use the butler.
build = subprocess.run(
    [
        "pipetask",
        "build",
        "-b",
        BUTLER_REPO,
        "-p",
        str(REPO_ROOT / "pipelines" / "drpQA.yaml"),
        "--show",
        "pipeline-graph",
    ],
    capture_output=True,
    text=True,
)
print(build.stdout or build.stderr)

expected = {"dmResiduals", "dmCombinedResiduals", "extractionQa", "extractionQaCombined", "imageQualityQa"}
missing = {label for label in expected if label not in build.stdout}
# The new output connection should appear in the resolved graph.
speciesDeclared = "iqQaSpeciesMetrics" in build.stdout

if build.returncode != 0:
    record("1 pipeline build", False, f"pipetask build exited {build.returncode}; see the output above")
else:
    notes = []
    if missing:
        # The graph resolved, which is what is being tested. A label absent from
        # the rendering is more likely a formatting difference than a missing task.
        notes.append(f"labels not found in output (check by eye): {sorted(missing)}")
    if not speciesDeclared:
        notes.append("iqQaSpeciesMetrics not seen in the graph -- confirm the new output connection")
    record(
        "1 pipeline build",
        True,
        "; ".join(notes) if notes else "graph resolves, all five labels and iqQaSpeciesMetrics present",
    )


## 2. Stack-dependent tests

`tests/test_dmResiduals.py` has never run against the real stack — only against a stub
with a hand-written `robustRms`. This is where a difference would show.

In [ ]:
tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", str(REPO_ROOT / "tests")],
    capture_output=True,
    text=True,
    cwd=str(REPO_ROOT),
)
print(tests.stdout[-4000:] or tests.stderr[-4000:])
record("2 tests", tests.returncode == 0, "pytest passed" if tests.returncode == 0 else "pytest failed, see above")

### 3a. Old ladder vs new registry, on stored numbers

**This section does not read `qaStatus`, and needs no pipeline run.** It pulls the measured
values out of `iqQaMetrics` -- `medFwhm`, `pctFlagged`, `medDxCenter`, `traceOnly`,
`seqName` -- and puts each row through two pure functions: the gating ladder transcribed
verbatim from `main`, and the new registry. Then it compares those two outputs *to each
other*.

So the collection is only a source of realistic numbers. This branch changes no
measurement code -- the diff to `imageQualityQa.py` touches imports, connections,
docstrings, the gating block and the new species output, and nothing that computes a
metric -- so those numbers are the same whichever branch produced them. **Any collection
with `iqQaMetrics` works, including one reduced months ago.**

Any disagreement is a real regression in the migration.


In [ ]:
from lsst.daf.butler import Butler

butler = Butler(BUTLER_REPO, collections=[COLLECTION], writeable=False)  # read-only

where = ""
bind = {}
if VISITS:
    where = "visit IN (visits)"
    bind = {"visits": list(VISITS)}

refs = sorted(
    set(butler.registry.queryDatasets("iqQaMetrics", where=where, bind=bind)),
    key=lambda ref: (ref.dataId.get("visit", 0), str(ref.dataId.get("arm", "")), ref.dataId.get("spectrograph", 0)),
)
print(f"{len(refs)} iqQaMetrics datasets in {COLLECTION}")

metrics = pd.concat([butler.get(ref) for ref in refs], ignore_index=True) if refs else pd.DataFrame()
if metrics.empty:
    skip("3a", "no iqQaMetrics in this collection; nothing to compare")
else:
    display(metrics[["visit", "arm", "spectrograph", "seqName", "medFwhm", "pctFlagged", "medDxCenter", "qaStatus"]].head(12))

In [ ]:
# The gating ladder exactly as it stood on `main`, transcribed for comparison.
# Do not "tidy" this: its value is being a faithful copy of what it replaced.
def legacyStatus(row, config):
    """Return the qaStatus that main's if/elif ladder would have produced."""
    medFwhm = row["medFwhm"]
    traceOnly = bool(row.get("traceOnly", False))
    pctFlagged = row["pctFlagged"]
    medDxCenter = row["medDxCenter"]
    seqName = row.get("seqName") or ""
    arm = row.get("arm") or ""

    fwhmStatus = "PASS"
    if not traceOnly and not np.isnan(medFwhm):
        if medFwhm >= config.fwhmFailThreshold:
            fwhmStatus = "FAIL"
        elif medFwhm >= config.fwhmWarnThreshold:
            fwhmStatus = "WARN"

    flagStatus = "PASS"
    if np.isfinite(pctFlagged):
        species = seqName.split(":", 1)[-1].strip() if ":" in seqName else ""
        compoundKey = f"{arm}:{species}" if species else ""
        warnThresh = config.flagRateWarnThreshold.get(compoundKey, config.flagRateWarnThreshold.get(arm, 15.0))
        failThresh = config.flagRateFailThreshold.get(compoundKey, config.flagRateFailThreshold.get(arm, 20.0))
        if pctFlagged >= failThresh:
            flagStatus = "FAIL"
        elif pctFlagged >= warnThresh:
            flagStatus = "WARN"

    dxStatus = "PASS"
    if np.isfinite(medDxCenter):
        absDx = abs(medDxCenter)
        if absDx >= config.dxCenterFailThreshold:
            dxStatus = "FAIL"
        elif absDx >= config.dxCenterWarnThreshold:
            dxStatus = "WARN"

    level = {"PASS": 0, "WARN": 1, "FAIL": 2}
    return max((fwhmStatus, flagStatus, dxStatus), key=lambda s: level[s])

In [ ]:
from pfs.drp.qa.imageQualityQa import ImageQualityQaConfig
from pfs.drp.qa.metrics.definitions import buildImageQualityRegistry
from pfs.drp.qa.metrics.registry import worstStatus

config = ImageQualityQaConfig()  # shipped defaults, as both branches use them
registry = buildImageQualityRegistry(config)


def registryStatus(row):
    """Return the qaStatus the new registry produces, mirroring the task."""
    arm = row.get("arm") or ""
    seqName = row.get("seqName") or ""
    species = seqName.split(":", 1)[-1].strip() if ":" in seqName else ""
    flagRateKeys = (f"{arm}:{species}" if species else "", arm)
    traceOnly = bool(row.get("traceOnly", False))
    return worstStatus(
        [
            None if traceOnly else registry.gate("medFwhm", row["medFwhm"]),
            registry.gate("pctFlagged", row["pctFlagged"], keys=flagRateKeys),
            registry.gate("medDxCenter", row["medDxCenter"]),
        ]
    )


if metrics.empty:
    skip("3a", "no iqQaMetrics to re-gate")
else:
    compare = metrics.copy()
    compare["legacy"] = compare.apply(lambda row: legacyStatus(row, config), axis=1)
    compare["registry"] = compare.apply(registryStatus, axis=1)
    disagree = compare[compare["legacy"] != compare["registry"]]

    if disagree.empty:
        record("3a", True, f"old ladder and registry agree on all {len(compare)} quanta")
    else:
        record("3a", False, f"{len(disagree)} of {len(compare)} quanta disagree -- see below")
        display(
            disagree[
                ["visit", "arm", "spectrograph", "seqName", "medFwhm", "pctFlagged", "medDxCenter", "legacy", "registry"]
            ]
        )

### 3b. Re-gated verdicts vs what is stored

Compares the registry's verdict against the `qaStatus` already in the collection.

**Read this one carefully.** It is a genuine parity check only if `COLLECTION` was produced
by `main`; if it came from this branch the comparison is circular and merely confirms
determinism. The notebook says which case it thinks it is in, but it cannot know for
certain — you do.

In [ ]:
if metrics.empty:
    skip("3b", "no iqQaMetrics to compare")
else:
    mismatch = compare[compare["registry"] != compare["qaStatus"]]
    hasSpecies = "iqQaSpeciesMetrics" in {dt.name for dt in butler.registry.queryDatasetTypes()}
    provenance = (
        "this collection appears to come from THIS branch (iqQaSpeciesMetrics is registered), "
        "so 3b confirms determinism rather than parity"
        if hasSpecies
        else "no iqQaSpeciesMetrics dataset type registered, consistent with a collection from main"
    )
    print(f"Note: {provenance}.")

    if mismatch.empty:
        record("3b", True, f"stored qaStatus matches the registry for all {len(compare)} quanta")
    else:
        record("3b", False, f"{len(mismatch)} quanta differ from the stored qaStatus")
        display(mismatch[["visit", "arm", "spectrograph", "seqName", "qaStatus", "registry"]])

In [ ]:
# 3c. If you did run the pipeline twice, diff the two collections directly.
if BASELINE_COLLECTION is None:
    skip("3c", "BASELINE_COLLECTION not set; skipping the two-collection diff")
else:
    baseButler = Butler(BUTLER_REPO, collections=[BASELINE_COLLECTION], writeable=False)
    baseRefs = set(baseButler.registry.queryDatasets("iqQaMetrics", where=where, bind=bind))
    baseline = pd.concat([baseButler.get(ref) for ref in baseRefs], ignore_index=True)

    key = ["visit", "arm", "spectrograph"]
    cols = ["qaStatus", "medFwhm", "medDxCenter", "pctFlagged"]
    left = baseline.set_index(key)[cols].sort_index()
    right = metrics.set_index(key)[cols].sort_index()
    common = left.index.intersection(right.index)
    diff = left.loc[common].compare(right.loc[common])

    if diff.empty:
        record("3c", True, f"{len(common)} quanta identical across the two collections")
    else:
        record("3c", False, f"{len(diff)} quanta differ -- see below")
        display(diff)

## 4. The new per-species dataset

`iqQaSpeciesMetrics` replaces the ragged `fitSpeciesXRms_<species>` columns. Two things to
confirm: the long frames concatenate to a stable schema, and `imageQualityLogQa.py` finds
them again — it read the removed columns, which review caught.

In [ ]:
from pfs.drp.qa.metrics.longFormat import LONG_COLUMNS

speciesRefs = set()
try:
    speciesRefs = set(butler.registry.queryDatasets("iqQaSpeciesMetrics", where=where, bind=bind))
except Exception as exc:  # noqa: BLE001
    print(f"query failed: {type(exc).__name__}: {exc}")

if not speciesRefs:
    skip("4 species dataset", "no iqQaSpeciesMetrics here; expected for a collection from main")
else:
    frames = [butler.get(ref) for ref in speciesRefs]
    combined = pd.concat(frames, ignore_index=True)
    schemaOk = tuple(combined.columns) == LONG_COLUMNS
    padded = combined["value"].isna().sum()
    print(f"{len(frames)} datasets, {len(combined)} rows, species seen: {sorted(combined['description'].unique())}")
    display(combined.head(12))

    if not schemaOk:
        record("4 species dataset", False, f"unexpected columns: {tuple(combined.columns)}")
    elif padded:
        record("4 species dataset", False, f"{padded} NaN values -- long format should not pad")
    else:
        record("4 species dataset", True, f"stable schema across {len(frames)} quanta, no NaN padding")

In [ ]:
# The report path that review found broken. Requires a visit present in the collection.
reportVisit = int(metrics["visit"].iloc[0]) if not metrics.empty else None
reportSpec = int(metrics["spectrograph"].iloc[0]) if not metrics.empty else None

if reportVisit is None:
    skip("4 report", "no visit available")
else:
    cmd = [
        sys.executable,
        str(REPO_ROOT / "bin.src" / "imageQualityLogQa.py"),
        "--butler", BUTLER_REPO,
        "--collection", COLLECTION,
        "--visit", str(reportVisit),
        "--spectrograph", str(reportSpec),
    ]
    print(" ".join(cmd), "\n")
    report = subprocess.run(cmd, capture_output=True, text=True)
    print(report.stdout[-4000:] or report.stderr[-4000:])
    if report.returncode != 0:
        record("4 report", False, f"imageQualityLogQa.py exited {report.returncode}")
    else:
        record(
            "4 report",
            True,
            "ran; CHECK BY EYE that per-species residuals appear -- an empty species "
            "section means the iqQaSpeciesMetrics merge is not firing",
        )

## 5. Plotting

The plotting functions moved to `pfs.drp.qa.plotting` and now take DataFrames and a
`DetectorGeometry` rather than a `DetectorMap`. Regenerating a figure from stored data
exercises that, without running the pipeline.

Set `SAVE_FIGURES = True` if you want them written to `OUTPUT_DIR`; otherwise nothing is
written to disk.

In [ ]:
SAVE_FIGURES = False

from pfs.drp.qa.plotting import DetectorGeometry, plot_detectormap_residuals

dataRefs = sorted(
    set(butler.registry.queryDatasets("dmQaResidualData", where=where, bind=bind)),
    key=lambda ref: (ref.dataId.get("visit", 0), str(ref.dataId.get("arm", ""))),
)
statRefs = {
    (ref.dataId.get("visit"), ref.dataId.get("arm"), ref.dataId.get("spectrograph")): ref
    for ref in butler.registry.queryDatasets("dmQaResidualStats", where=where, bind=bind)
}

if not dataRefs:
    skip("5 plotting", "no dmQaResidualData in this collection")
else:
    ref = dataRefs[0]
    dataId = ref.dataId
    statRef = statRefs.get((dataId.get("visit"), dataId.get("arm"), dataId.get("spectrograph")))
    arcData = butler.get(ref)
    visitStats = butler.get(statRef) if statRef is not None else None

    if visitStats is None:
        skip("5 plotting", "matching dmQaResidualStats not found")
    else:
        detectorMap = butler.get("detectorMap", dataId=dataId)
        geometry = DetectorGeometry.fromDetectorMap(detectorMap)
        print(f"{dict(dataId.mapping)} -> {geometry}")

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            figure = plot_detectormap_residuals(arcData, visitStats, geometry)

        if figure is None:
            record("5 plotting", False, "plot_detectormap_residuals returned None")
        else:
            record("5 plotting", True, "figure drawn from stored data via DetectorGeometry")
            if SAVE_FIGURES:
                Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
                out = Path(OUTPUT_DIR) / "dmResiduals.png"
                figure.savefig(out, dpi=120)
                print(f"wrote {out}")
            display(figure)

## 6. The golden visit set

If this collection covers the Run25 stable sequence, every `known_good` detector should
report PASS. **A failure here is the most interesting result in the notebook**: either a
threshold is wrong, or the "assumed stable" premise does not hold — and either way you want
to know before thresholds are re-derived from those visits.

In [ ]:
from pfs.drp.qa.metrics.goldenVisits import loadGoldenVisits

golden = loadGoldenVisits(REPO_ROOT / "tests" / "data" / "goldenVisits.yaml")
print(f"{len(golden.knownGood)} known_good entries, {len(golden.knownBad)} known_bad "
      f"({len(golden.confirmedBad)} confirmed)")

if metrics.empty:
    skip("6 golden set", "no iqQaMetrics to check")
else:
    rows = []
    for row in metrics.to_dict("records"):
        expected = golden.expectationFor(
            int(row["visit"]),
            arm=row.get("arm"),
            spectrograph=int(row["spectrograph"]) if pd.notna(row.get("spectrograph")) else None,
            seqType=row.get("seqName"),
        )
        if expected is None:
            continue  # not in the golden set: no expectation, which is not a PASS
        rows.append({**row, "expected": expected, "actual": row["qaStatus"]})

    checked = pd.DataFrame(rows)
    if checked.empty:
        skip("6 golden set", "none of these visits are in the golden set")
    else:
        wrong = checked[checked["expected"] != checked["actual"]]
        print(f"{len(checked)} quanta carry a golden-set expectation")
        if wrong.empty:
            record("6 golden set", True, f"all {len(checked)} quanta match their expectation")
        else:
            record("6 golden set", False, f"{len(wrong)} quanta do not match -- see below")
            display(wrong[["visit", "arm", "spectrograph", "seqName", "expected", "actual",
                           "medFwhm", "pctFlagged", "medDxCenter"]])

## 7. Threshold calibration CLI

Reads the collection and prints suggestions. It writes nothing. A non-zero exit is
meaningful, not a crash: it refuses to hand you thresholds it cannot stand behind — too
few samples, or known-bad data that does not cross the suggested FAIL.

In [ ]:
cmd = [
    sys.executable,
    str(REPO_ROOT / "bin.src" / "calibrateQaThresholds.py"),
    "-b", BUTLER_REPO,
    "-c", COLLECTION,
    "--metric", "medFwhm",
    "--metric", "pctFlagged",
    "--group-by", "arm",
]
print(" ".join(cmd), "\n")
calib = subprocess.run(cmd, capture_output=True, text=True)
print(calib.stdout)
print(calib.stderr, file=sys.stderr)

record(
    "7 threshold CLI",
    calib.returncode in (0, 1),
    f"exited {calib.returncode} "
    + ("(suggestions produced)" if calib.returncode == 0 else "(refused to stand behind the numbers -- read the reason above)"),
)

## Summary

In [ ]:
summary = pd.DataFrame(
    [
        {"section": name, "result": "PASS" if ok else ("SKIP" if ok is None else "FAIL"), "detail": message}
        for name, (ok, message) in RESULTS.items()
    ]
)
display(summary)

failed = [name for name, (ok, _) in RESULTS.items() if ok is False]
if failed:
    print(f"\nFAILED: {', '.join(failed)}")
    print("Sections 1, 2 and 3 are the blocking ones; 4-7 are quality checks.")
else:
    print("\nNothing failed. Sections marked SKIP had no data to work with.")